# Prepping Data Challenge - Week 19

## Import Libraries

In [1]:
import numpy as np
import pandas as pd

## Import Data

In [2]:
YEARS = [2018, 2019, 2020, 2021, 2022]
quarterly_sales = [pd.read_excel('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_19_Sales_Data.xlsx', sheet_name=str(YEAR)) for YEAR in YEARS]


### Change the sales values to numbers, aggregate to year and combine 

#### Change `Sales` and `Profits` to numerical

In [3]:
quarterly_sales[3]

,Unnamed: 0,Sales,Profits
0,Q1,"3,701.345K","1,301.234K"
1,Q2,"4,000.122K","1,412.678K"
2,Q3,4.41M,"1,700.346K"
3,Q4,"4,897.312K","1,990K"


The problem with this method is, that we want to make it fool proof for cases, where the format deviates

In [4]:
def money_strings_to_numeric(amount):

    if isinstance(amount, int):
        return amount
    
    amount = str(amount)

    # Extract the order of magnitude (M for Million, K for Thousand)
    magnitude = amount[-1]
    amount = amount[:-1]

    # Split the amount at the magnitude threshold
    if '.' not in amount:
        amount = amount + '.0'
    
    lead, trail = amount.split('.')
    
    if magnitude == 'K':
        LENGTH_OF_TRAIL = 3
    elif magnitude == 'M':
        LENGTH_OF_TRAIL = 6

    # Add trailing zeros based on magnitude if necessary
    trail = trail.ljust(LENGTH_OF_TRAIL, '0')

    # Combine the prefix and suffix
    lead = lead.replace(',', '')
    final_amount = lead + trail      # Concatenate, since amount is still a skill

    return int(final_amount)

In [5]:
for yearly_sales in quarterly_sales:
    yearly_sales['Profits'] = yearly_sales['Profits'].apply(money_strings_to_numeric)
    yearly_sales['Sales'] = yearly_sales['Sales'].apply(money_strings_to_numeric)

In [6]:
# Add a year column
for i in range(len(quarterly_sales)):
    quarterly_sales[i]['Year'] = YEARS[i]

In [7]:
# Aggregate to year
yearly_sales = [df.agg({'Sales' : 'sum', 'Profits' : 'sum', 'Year' : 'mean'}).to_frame().T.set_index('Year') for df in quarterly_sales]

In [8]:
# Cast as int
output = pd.concat(yearly_sales).astype(int)
output.index = output.index.astype(int)

In [9]:
output

,Sales,Profits
Year,,
2018,18261030,5470497
2019,18505267,5400667
2020,16111418,5410007
2021,17008779,6404258
2022,18099644,5766245


In [10]:
# Save Results
output.to_csv('D:01_Projekt_Portfolio/data_prepping_challenges/outputs/19_yearly_sales.csv')